# Inspect the dynamics RNN

Choose `z` or `z_psi` to compare the supplied checkpoint with a small dataset sample.
This inspection is not a held-out accuracy estimate. New training runs record their split in `training.json`.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/model/rnn_loader.py").is_file()
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from model.training.train import read_csv, prepare_sequences
import torch

from model.rnn_loader import load_rnn_model, checkpoint_parameters
from policy.lib.droneEnv import PARAMS as Z_PARAMS
from policy.lib.droneEnvRotation import PARAMS as Z_PSI_PARAMS

MODE = "z"
MAX_TRAJECTORIES = 32
CHECKPOINT = PROJECT_ROOT / "src/model/checkpoints" / f"rnn_{MODE}.pt"

DATASETS = {
    "z": {
        "folder": "initial_data",
        "inputs": "bdd_in_mat_05.csv",
        "outputs": "bdd_out_mat_05.csv",
        "input_format": "controls",
        "output_format": "selected",
        "defaults": Z_PARAMS,
        "channels": ["az"],
    },
    "z_psi": {
        "folder": "generated_data",
        "inputs": "db_regime6_helico_input.csv",
        "outputs": "db_regime6_helico_output.csv",
        "input_format": "motors",
        "output_format": "plant",
        "defaults": Z_PSI_PARAMS,
        "channels": ["az", "wz_dot"],
    },
}
settings = DATASETS[MODE]
folder = PROJECT_ROOT / "data" / settings["folder"]
inputs, targets = prepare_sequences(
    read_csv(folder / settings["inputs"], MAX_TRAJECTORIES),
    read_csv(folder / settings["outputs"], MAX_TRAJECTORIES),
    MODE,
    settings["input_format"],
    settings["output_format"],
)
model = load_rnn_model(CHECKPOINT)
params = checkpoint_parameters(model, settings["defaults"], MODE)
print(model)
print(f"{len(inputs)} trajectories, {inputs.shape[1]} samples each")


The recurrent state resets between trajectories. Predictions are converted back to physical accelerations before comparison.


In [ ]:
input_mean = np.asarray(params.mean_u, dtype=np.float32)
input_scale = np.asarray(params.sigma_u, dtype=np.float32)
input_scale = np.where(input_scale == 0, 1, input_scale)
output_mean = np.array([getattr(params, f"mean_{name}") for name in settings["channels"]])
output_scale = np.array([getattr(params, f"sigma_{name}") for name in settings["channels"]])

normalized = torch.from_numpy((inputs - input_mean) / input_scale)
with torch.inference_mode():
    predictions, _ = model(normalized)
predictions = predictions.numpy() * output_scale + output_mean

for index, name in enumerate(settings["channels"]):
    error = predictions[..., index] - targets[..., index]
    print(f"{name}: MAE {np.abs(error).mean():.4g}, RMSE {np.sqrt(np.mean(error**2)):.4g}")


In [ ]:
trajectory = 0
channels = settings["channels"]
fig, axes = plt.subplots(len(channels), 1, figsize=(10, 3 * len(channels)), squeeze=False)
time = np.arange(inputs.shape[1]) * params.dt
for index, name in enumerate(channels):
    axis = axes[index, 0]
    axis.plot(time, targets[trajectory, :, index], label="Plant")
    axis.plot(time, predictions[trajectory, :, index], label="RNN")
    axis.set_ylabel(name)
    axis.legend()
    axis.grid(alpha=0.3)
axes[-1, 0].set_xlabel("Time (s)")
fig.tight_layout()
